# NIST Parylene-C Palace CPW benchmark

**Objective.** Model the complete 6.5 mm, air-filled H2O-chip CPW in Palace and compare the full complex S matrix first to the published ANSYS Q2D cascade, then to the calibrated measurement.

**Hypothesis.** The 0.2 µm Pt model should approach Q2D after matching all nine line sections. The capped screening run uses explicit 50-ohm CPW lumped ports; their termination reflection must be separated from device/model error. The separately labelled 0.405 µm Pt case is the appropriate measurement comparison.

The default execution validates the stored references and constructs the model without meshing or submitting. Set `GSIM_EIC_RUN_MESH=1` for local meshing or `GSIM_EIC_RUN_CLOUD=1` for the opt-in three-frequency cloud screening run. Every Palace execution is blocked unless the finalized mesh contains at most 110,000 tetrahedra.

In [ ]:
# Copyright 2026 GDSFactory

from __future__ import annotations

import importlib.metadata
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from gsim.palace.benchmarks import (
    from_palace_sparams,
    interpolate_sparameters,
    maximum_singular_value,
    power_loss_fraction,
    reciprocity_error,
    sparameter_error_summary,
)
from gsim.palace.benchmarks.eic_nist import (
    NIST_CAPPED_MESH_SIZE_UM,
    NIST_MAX_TETRAHEDRA,
    NIST_MEASUREMENT_DOI,
    NIST_SIMULATION_DOI,
    build_nist_component,
    build_nist_stack,
    cascade_nist_rlcg,
    load_nist_air_reference,
    make_nist_simulation,
    nist_section_edges_um,
    require_nist_tetrahedron_budget,
)


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from within the gsim repository")


repo_root = find_repo_root()
data_dir = repo_root / "tests" / "data" / "eic" / "nist"
run_cloud = os.getenv("GSIM_EIC_RUN_CLOUD") == "1"
run_mesh = os.getenv("GSIM_EIC_RUN_MESH") == "1" or run_cloud
model_case = os.getenv("GSIM_EIC_NIST_CASE", "ansys").lower()
if model_case not in {"ansys", "measurement"}:
    raise ValueError("GSIM_EIC_NIST_CASE must be 'ansys' or 'measurement'")
print(
    {
        "gsim": importlib.metadata.version("gsim"),
        "case": model_case,
        "mesh": run_mesh,
        "cloud": run_cloud,
    }
)

## Published references

The inputs are from NIST datasets [10.18434/mds2-2817](https://doi.org/10.18434/mds2-2817) (simulation) and [10.18434/mds2-2808](https://doi.org/10.18434/mds2-2808) (measurement). The final RLCG CSVs reproduce `constructSfromSim.m`. The stored MAT result was created earlier than those CSV revisions, so the regression records their small, bounded difference instead of claiming bit identity.

In [ ]:
reference = load_nist_air_reference(data_dir / "air_data_vs_sim.npz")
reconstructed = cascade_nist_rlcg(data_dir)
archive_revision_delta = np.abs(reconstructed.s - reference.simulation.s)
np.testing.assert_allclose(np.max(archive_revision_delta), 0.0017645356, rtol=1e-6)

print("simulation DOI:", NIST_SIMULATION_DOI)
print("measurement DOI:", NIST_MEASUREMENT_DOI)
print(f"final-CSV vs stored-MAT max |delta S|={np.max(archive_revision_delta):.8f}")
print(f"Q2D reciprocity error={reciprocity_error(reference.simulation):.3g}")
print(f"Q2D largest singular value={maximum_singular_value(reference.simulation):.9f}")
print(f"measurement reciprocity error={reciprocity_error(reference.measurement):.4g}")
print(
    f"measurement largest singular value={maximum_singular_value(reference.measurement):.6f}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
for data, style, label_prefix in (
    (reference.simulation, "-", "Q2D"),
    (reference.measurement, "--", "measured"),
):
    band = (data.frequency_hz >= 0.1e9) & (data.frequency_hz <= 20e9)
    frequency_ghz = data.frequency_hz[band] / 1e9
    for axis, (label, row, column) in zip(
        axes, (("S11", 0, 0), ("S21", 1, 0)), strict=True
    ):
        axis.plot(
            frequency_ghz,
            20 * np.log10(np.abs(data.s[band, row, column])),
            style,
            label=f"{label_prefix} {label}",
        )
for axis, title in zip(axes, ("Reflection", "Transmission"), strict=True):
    axis.set(xlabel="Frequency (GHz)", ylabel="Magnitude (dB)", title=title)
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()

## Complete geometry and material cases

The nine-section sequence is `NP–YP–AirPDMS–PDMS–AirChannel–PDMS–AirPDMS–YP–NP`. The centered 213 µm air channel appears in both AirPDMS transitions and the central channel, overriding the surrounding PDMS. The CPW is 50/2.5/200 µm signal/gap/ground on 500 µm fused silica, with 6.67 µm Parylene C.

`ansys` selects the source model's 0.2 µm Pt and Q2D as its target. `measurement` selects the measured 0.405 µm Pt thickness and calibrated data; set it with `GSIM_EIC_NIST_CASE=measurement`.

In [ ]:
component = build_nist_component()
edges = nist_section_edges_um()
ansys_stack = build_nist_stack(platinum_thickness_um=0.2)
measurement_stack = build_nist_stack(platinum_thickness_um=0.405)
assert ansys_stack.validate_stack().valid
assert measurement_stack.validate_stack().valid
np.testing.assert_allclose(component.bbox_np(), [[-3250.25, -682.5], [3250.25, 682.5]])

print("section edges (um):", edges)
print("total line length (um):", edges[-1] - edges[0])
print("ANSYS Pt thickness (um):", ansys_stack.layers["platinum"].thickness)
print("measured Pt thickness (um):", measurement_stack.layers["platinum"].thickness)

In [ ]:
platinum_thickness_um = 0.2 if model_case == "ansys" else 0.405
target_reference = (
    reference.simulation if model_case == "ansys" else reference.measurement
)
simulation = make_nist_simulation(
    repo_root / f"palace-sim-eic-nist-{model_case}",
    platinum_thickness_um=platinum_thickness_um,
    num_points=3,
    adaptive_tol=0.0,
    adaptive_max_samples=1,
)
assert simulation.validate_config().valid
palace_result = None
mesh_result = None
print(
    f"Configured {model_case} case: Pt={platinum_thickness_um} um, two 50-ohm CPW lumped ports."
)

## Optional mesh and run

The opt-in mesh retains the full 6.5005 mm by 1.365 mm physical device and adds 50 µm longitudinal airbox margin outside each lumped port. Same-coefficient geometry partitions force two 1.25 µm transverse spans across every 2.5 µm gap at the metal plane while allowing wavelength-scale longitudinal and bulk elements. This is not isotropic 1.25 µm convergence. `planar_conductors=False` keeps Pt finite, and both CPW lumped ports are excited so one job emits S11, S12, S21, and S22. Submit the two material cases separately; do not interpret material loss until reciprocity, passivity, port, and mesh convergence have been checked.

In [ ]:
if run_mesh:
    mesh_result = simulation.mesh(
        preset="coarse",
        refined_mesh_size=NIST_CAPPED_MESH_SIZE_UM,
        max_mesh_size=NIST_CAPPED_MESH_SIZE_UM,
        planar_conductors=False,
        verbose=False,
    )
    tetrahedra = require_nist_tetrahedron_budget(mesh_result)
    config_path = simulation.write_config()
    validation = simulation.validate_mesh()
    config = json.loads(config_path.read_text())
    conductivity_boundaries = config["Boundaries"]["Conductivity"]
    lumped_ports = config["Boundaries"]["LumpedPort"]
    assert len(conductivity_boundaries) == 2
    np.testing.assert_allclose(
        [entry["Thickness"] for entry in conductivity_boundaries],
        platinum_thickness_um,
    )
    assert len(lumped_ports) == 2
    assert all(port["R"] == 50.0 for port in lumped_ports)
    assert all(len(port["Elements"]) == 2 for port in lumped_ports)
    assert tetrahedra <= NIST_MAX_TETRAHEDRA
    print(validation)
    print(f"tetrahedra: {tetrahedra:,} / {NIST_MAX_TETRAHEDRA:,}")
    print(mesh_result.mesh_stats)
    print("conductivity boundaries:", conductivity_boundaries)
else:
    print("Mesh skipped; set GSIM_EIC_RUN_MESH=1 to reproduce local validation.")

In [ ]:
if run_cloud:
    require_nist_tetrahedron_budget(mesh_result)
    palace_result = simulation.run(check_cache=True, verbose="status")
    print("cloud job id:", simulation._job_id)
else:
    print("Cloud submission skipped; set GSIM_EIC_RUN_CLOUD=1 explicitly.")

In [ ]:
if palace_result is not None:
    palace = from_palace_sparams(palace_result)
    aligned_reference = interpolate_sparameters(target_reference, palace.frequency_hz)
    parity = sparameter_error_summary(aligned_reference, palace)
    print(json.dumps(parity, indent=2))
    print(f"Palace reciprocity error={reciprocity_error(palace):.3g}")
    print(f"Palace largest singular value={maximum_singular_value(palace):.6f}")
    print(
        "Palace unscattered power range:",
        power_loss_fraction(palace).min(),
        power_loss_fraction(palace).max(),
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
    for data, style, prefix in (
        (aligned_reference, "-", "reference"),
        (palace, "--", "Palace"),
    ):
        for label, row, column in (
            ("S11", 0, 0),
            ("S12", 0, 1),
            ("S21", 1, 0),
            ("S22", 1, 1),
        ):
            values = data.s[:, row, column]
            axes[0].plot(
                data.frequency_hz / 1e9,
                20 * np.log10(np.abs(values)),
                style,
                label=f"{prefix} {label}",
            )
            axes[1].plot(
                data.frequency_hz / 1e9,
                np.unwrap(np.angle(values)) * 180 / np.pi,
                style,
                label=f"{prefix} {label}",
            )
    axes[0].set(xlabel="Frequency (GHz)", ylabel="Magnitude (dB)")
    axes[1].set(xlabel="Frequency (GHz)", ylabel="Unwrapped phase (degrees)")
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend(ncol=2)
    fig.tight_layout()
else:
    print("Parity table will be emitted after an opt-in Palace run completes.")

## Findings and decision log

- The curated arrays preserve all four complex entries and both published frequency grids.
- The final archive CSVs reproduce the older stored Q2D cascade to a maximum `|delta S|` of 0.001765; provenance dates explain the nonzero revision delta.
- The Palace model is the complete 6.5005 mm measured section sequence, not a shortened or periodic surrogate.
- The first capped cloud input had 14,274 nodes and 71,370 tetrahedra. Job `01a05c94-afbd-7c42-a72b-4a54d30004b0` (`71ecf571...`) completed all six 3D solves in 12.9 s, converged in 9–29 GMRES iterations, and was reciprocal to `1.54e-7`, but its end absorbing boundaries were flush with the lumped ports. Its S21 values of -29.7, -65.3, and -79.7 dB fail the NIST comparison, so this result is rejected rather than treated as parity.
- The corrected model follows the repository's known-good lumped-CPW setup by adding 50 um of longitudinal airbox margin. Its finalized local mesh has 14,394 nodes and 72,768 tetrahedra, leaving 37,232 tetrahedra of headroom. It has zero invalid SICN elements, all six gap/conductor lines span the complete device, and every lumped-port triangle is a valid two-sided internal tetrahedron face. This corrected input has not been submitted.
- The capped model remains the complete 3D device. Same-coefficient mesh partitions force two 1.25 um transverse cells across each gap without changing material coefficients; the scalar 150 um setting controls longitudinal/bulk size and is not a claim of isotropic refinement.
- Initial job `01a05b6c-d591-7bf0-8f35-ee5477d7e4a7` exposed and led to the airbox/port-topology fix. Repaired job `01a05b71-e7ff-70c0-a69a-ccda87b54375` (`5309580e...`, Palace `4930e88`) had 273,551 tetrahedra and failed at the cloud maximum runtime. Its final log still ended in the first 0.1 GHz numeric wave-port boundary-mode solve; no 3D frequency solve began.
- The replacement uses two explicit 50-ohm CPW lumped ports, eliminating that numeric boundary eigensolve. Lumped-port reflection is part of the screening-model uncertainty and must not be misattributed to the NIST device.
- The ANSYS and measurement thickness cases remain intentionally separate. The hard 110,000-tetrahedron gate is checked after meshing and again immediately before submission. Solver, port, and mesh convergence are required before interpreting agreement or material loss.